# 02 — Daily and weekly rhythms
 **RQ1:** Do hourly rental patterns differ for registered and casual users, and for working vs non-working days?

 **RQ2:** Which days and hours are busiest?

 **Read carefully:** `workingday=0` includes weekends **and public holidays**. Lines represent mean observed rental counts per hour, not unique cyclists. Different panels use different y-axis ranges to keep both patterns legible.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

hour = pd.read_csv('../data/processed/bike_hour_clean.csv')
if 'day_name' not in hour.columns:
    hour['day_name'] = pd.to_datetime(hour['dteday']).dt.day_name()
Path('../output/figures').mkdir(parents=True, exist_ok=True)

COLORS = {'registered': '#2563EB', 'casual': '#E69F00'}
hourly_avg = hour.groupby(['workingday','hr'])[['registered','casual']].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
for ax, key, label in zip(axes, ['registered','casual'], ['Registered users','Casual users']):
    color = COLORS[key]
    for workingday, line_style, day_label in [(1,'-','Working days'), (0,'--','Days off / holidays')]:
        part = hourly_avg[hourly_avg['workingday']==workingday]
        ax.plot(part['hr'],part[key],linestyle=line_style, linewidth=2.6,
                color=color,label=day_label)
        pk = part.loc[part[key].idxmax()]
        ax.scatter([pk['hr']],[pk[key]],color=color,s=25,zorder=4)
        print(f'{label} | {day_label}: peak {int(pk["hr"]):02d}:00, '
              f'{pk[key]:.1f} rentals/hour')
    ax.set_title(label,fontweight='bold')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Average hourly rentals')
    ax.set_xticks(range(0,24,3))
    ax.set_xlim(0,23)
    ax.set_ylim(bottom=0)
    ax.grid(axis='y',alpha=.16)
    ax.legend(frameon=False,loc='upper left', fontsize=9)

fig.suptitle('Two weekday commute peaks; a broader midday pattern on days off',
             fontsize=14,fontweight='bold',y=1.04)
fig.text(.01,-.04,'Note: days off include weekends and holidays; panel scales differ. '
         'UCI / Capital Bikeshare, 2011–2012.', fontsize=9,color='#555555')
fig.tight_layout()
fig.savefig('../output/figures/RQ1_daily_rhythm.png',dpi=200,bbox_inches='tight')
plt.show()

In [ ]:
days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekly = (hour.groupby(['day_name','hr'])['cnt'].mean().unstack()
          .reindex(index=days, columns=range(24)))
peak_day,peak_hour = weekly.stack().idxmax()
peak_value = weekly.stack().max()

fig,ax=plt.subplots(figsize=(13,5.4))
sns.heatmap(weekly,cmap='YlOrRd',linewidths=.3,ax=ax,
            cbar_kws={'label':'Average rentals per observed hour'})
ax.set_title('Weekday rush hours contrast with weekend midday demand',
             fontsize=14,fontweight='bold',pad=12)
ax.set_xlabel('Hour of day');ax.set_ylabel('Day of week')
ax.tick_params(axis='x',rotation=0);ax.tick_params(axis='y',rotation=0)
ax.add_patch(plt.Rectangle((peak_hour,days.index(peak_day)),1,1,fill=False,
                           edgecolor='#111111',linewidth=2.6))
fig.text(.01,-.01,f'Highest observed day–hour mean: {peak_day} {peak_hour:02d}:00 '
         f'({peak_value:.0f} rentals). Source: UCI / Capital Bikeshare, 2011–2012.',
         fontsize=9,color='#555555')
fig.tight_layout()
fig.savefig('../output/figures/RQ2_weekly_heatmap.png',dpi=200,bbox_inches='tight')
plt.show()
print(f'Weekly mean peak: {peak_day} at {peak_hour:02d}:00, {peak_value:.1f} rentals/hour.')

### Interpretation and limits
 Morning and evening peaks on working days are **consistent with** commuting, but the dataset does not record trip purpose. The two figures partly overlap in subject: in the final presentation we may select only the clearest or use one as a supporting figure.